El siguiente Notebook representa el flujo de trabajo en el framework Flex para efectuar el ataque conocido como "model replacement".

Primero se carga la base de datos a utilizar, en este caso Mnist.(para mayor información consultar la documentación oficial de la tecnología.)

In [46]:
from process_data import *
from copy import deepcopy

flex_dataset, server_id = load_and_preprocess_horizontal(dataname="mnist", trasnform=False, nodes=10)

torch.Size([60000, 28, 28])


A continuación, se define la arquitectura de los modelos locales de los clientes. Para el presente ejemplo se trabaja con modelos neuronales de pytorch.

Se utiliza el módulo networks_models, quien contiene una serie de modelos neuronales auxiliares de pytorch, para el trabajo con las bases de datos anteriormente mencionadas. Además se utiliza el módulo auxiliar networks_execution, que define la ejecución del entrenamiento y otros detalles de estos modelos.

Para establecer un modelo personalizado, ir a la documentación de Flex.

In [47]:
from networks_models import *
from networks_execution import *
from flex.pool import init_server_model
from flex.model import FlexModel

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.cuda.is_available()
    else "cpu"
)

net_config = ExecutionNetwork()


@init_server_model
def build_server_model():
    server_flex_model = FlexModel()
    criterion, model, optimizer = net_config.for_fd_server_model_config()
    server_flex_model["model"] = model.to(device)
    # Required to store this for later stages of the FL training process
    server_flex_model["criterion"] = criterion
    server_flex_model["optimizer_func"] = optimizer
    server_flex_model["optimizer_kwargs"] = {}
    return server_flex_model


A continuacion se define la arquitectura del modelo federado

In [48]:
from flex.pool import FlexPool

flex_pool = FlexPool.client_server_pool(
    fed_dataset=flex_dataset, server_id=server_id, init_func=build_server_model
)

clients = flex_pool.clients
servers = flex_pool.servers
aggregators = flex_pool.aggregators

Se define la funcion para desplegar el modelo global de cada cliente, además se intercepta el un cliente para efectuar el ataque y se inicializa una instancia de la clase que contiene el ataque.

In [49]:
from attack.model_replacement_attack import ModelReplacement as mr
from flex.pool import deploy_server_model
import copy

rounds = 0
round_global_model = None
norm_bound = 1
attack = mr(norm_bound)


@deploy_server_model
def copy_server_model_to_clients(server_flex_model: FlexModel):
    new_model = copy.deepcopy(server_flex_model)
    global round_global_model
    if rounds == 0:
        attack.set_model(copy.deepcopy(server_flex_model["model"]))

    return new_model

Se define las rondas de entrenamiento local del cliente.

In [50]:
def train(client_flex_model: FlexModel, client_data: Dataset):
    print(np.array(client_data.X_data).shape)
    train_dataset = client_data.to_torchvision_dataset(transform=mnist_transform())
    client_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True)

    model = client_flex_model['model']
    model = model.to(device)

    client_flex_model["previous_model"] = deepcopy(
        model
    )
    optimizer = client_flex_model["optimizer_func"]
    criterion = client_flex_model["criterion"]

    net_config.train_network(local_epochs=1, criterion=criterion, optimizer=optimizer, momentum=0.9, lr=0.005,
                             trainloader=client_dataloader, testloader=None,
                             model=model)

    return client_flex_model

Se define el entrenamiento del modelo adversario que sustituira un modelo benigno.

In [51]:
from flexclash.model import model_poison_agregator, model_poisoner

list_of_mr = []
@model_poisoner
def model_replacement(client_flex_model: FlexModel):
    if client_flex_model.actor_id in list_of_mr:
        clients_data_sizes = {
            client.actor_id: len(client.data.X_data) for client in selected_test_clients
        }
        attack.calculate_scaling_factor(selected_test_clients, client_data_sizes=clients_data_sizes)
        client_flex_model["model"] = attack.poison_model_update(client_flex_model["model"])

    return client_flex_model

Se efectúa la agregación del modelo federado

In [52]:
from flex.pool import collect_client_diff_weights_pt
from flex.pool import fed_avg
from flex.pool import set_aggregated_diff_weights_pt

#pool.aggregators.map(collect_client_diff_weights_pt, selected_test_clients)
#pool.aggregators.map(fed_avg)
#pool.aggregators.map(set_aggregated_diff_weights_pt, pool.servers)

Se evalúa el modelo federado

In [53]:
def evaluate_global_model(server_flex_model: FlexModel, test_data: Dataset):
    model = server_flex_model["model"]
    model.eval()
    test_loss = 0
    test_acc = 0
    total_count = 0
    model = model.to(device)

    criterion = server_flex_model["criterion"]
    # get test data as a torchvision object
    test_dataset = test_data.to_torchvision_dataset(transform=mnist_transform())
    test_dataloader = DataLoader(
        test_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=False
    )
    losses = []
    with torch.no_grad():
        for data, target in tqdm(test_dataloader):
            total_count += target.size(0)
            data, target = data.to(device), target.to(device)
            output = model(data)
            losses.append(criterion(output, target).item())
            pred = output.data.max(1, keepdim=True)[1]
            test_acc += pred.eq(target.data.view_as(pred)).long().cpu().sum().item()

    test_loss = sum(losses) / len(losses)
    test_acc /= total_count

    return test_loss, test_acc


Para limpiar los modelos en memoria. Opcional

In [54]:
def clean_up_models(client_model: FlexModel, _):
    import gc

    client_model.clear()
    gc.collect()

Se definen las rondas de entrenamiento del modelo federado. Este método engloba los anteriores. Además en este caso se describe en cada momento como se enctua el ataque. 

In [55]:
from auxiliar.extract_images import extract_digits_from_directory
from flexclash.pool import  central_differential_privacy

images_path = "C:\\Users\\Adrian\\PycharmProjects\\pythonProject\\generated_images"

def train_n_rounds(n_rounds=2, clients_per_round=10):
    flex_pool = FlexPool.client_server_pool(
    fed_dataset=flex_dataset, server_id=server_id, init_func=build_server_model
)
    global rounds
    global round_global_model
    for i in range(n_rounds):
        print(f"\nRunning round: {i + 1} of {n_rounds}")
        selected_clients_pool = flex_pool.clients.select(clients_per_round)
        selected_clients = selected_clients_pool.clients
        print("Selected clients:", len(selected_clients))
        print(f"Selected clients for this round: {len(selected_clients)}")
        # Deploy the server model to the selected clients
        flex_pool.servers.map(copy_server_model_to_clients, selected_clients)
        # Extract gan adversarial images for the trainning
        gan_images = extract_digits_from_directory(images_path)
        # Adv trainning
        attack.create_adversarial_model(gan_images)
        replaced_model_clients = selected_clients_pool.select(lambda actor_id, set_of_roles: actor_id in list_of_mr)
        replaced_model_clients = replaced_model_clients.clients
        # Bening trainning
        normal_clientss = selected_clients_pool.select(lambda actor_id, set_of_roles: actor_id not in list_of_mr)
        normal_clientss = normal_clientss.clients
        # Execute attack at this point with replaced_model_clients
        replaced_model_clients.map(model_replacement)
        # Each selected client trains her model
        normal_clientss.map(train)
        # The aggregador collects weights from the selected clients and aggregates them
        flex_pool.aggregators.map(collect_client_diff_weights_pt, selected_clients)
        flex_pool.aggregators.map(central_differential_privacy)

        # The aggregator send its aggregated weights to the server
        flex_pool.aggregators.map(set_aggregated_diff_weights_pt, flex_pool.servers)
        metrics = flex_pool.servers.map(evaluate_global_model)
        loss, acc = metrics[0]
        print(f"Global accuracy Server: Test acc: {acc:.4f}, test loss: {loss:.4f}")

        # Optional
        selected_clients.map(clean_up_models)

In [56]:
train_n_rounds(n_rounds=1, clients_per_round=5)


Running round: 1 of 1
Selected clients: 5
Selected clients for this round: 5
Injecting Over: Bad Imgs: 1494. Clean Imgs: 5976. Epsilon: 0.2
(6000, 28, 28)


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:02<00:00,  8.14it/s]


(6000, 28, 28)


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:02<00:00,  8.10it/s]


(6000, 28, 28)


...

(6000, 28, 28)


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:02<00:00,  8.45it/s]


(6000, 28, 28)


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:04<00:00,  8.81it/s]


Global accuracy Server: Test acc: 0.5867, test loss: 1.2486
